In [35]:
df = pd.read_csv(r"C:\Users\Muviniish\Machine Learning\CW1\Dataset\customer_shopping_behavior.csv")
#Copy Dataset
clean_df = df.copy()
noise_count = (
    (clean_df["Age"] < 18).sum() +
    (clean_df["Purchase Amount (USD)"] < 0).sum() +
    ((clean_df["Review Rating"] < 1) | (clean_df["Review Rating"] > 5)).sum() +
    (clean_df["Previous Purchases"] < 0).sum()
)
print("Before")
print("Duplicates:", clean_df.duplicated().sum())
print("Missing Values:", clean_df.isnull().sum().sum())
print("Total Noise Values:", noise_count)
print("Dataset Size:", clean_df.shape)
#Remove Duplicates
clean_df.drop_duplicates(inplace=True)
#Remove Noise
clean_df.loc[clean_df["Age"] < 18, "Age"] = None
clean_df.loc[clean_df["Purchase Amount (USD)"] < 0, "Purchase Amount (USD)"] = None
clean_df.loc[(clean_df["Review Rating"] < 1) | (clean_df["Review Rating"] > 5), "Review Rating"] = None
clean_df.loc[clean_df["Previous Purchases"] < 0, "Previous Purchases"] = None
#Remove Missing Values for Numerical
for col in ["Purchase Amount (USD)", "Review Rating", "Previous Purchases"]:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())
#Remove Missing Values for Categorical
clean_df["Size"] = clean_df["Size"].fillna(clean_df["Size"].mode()[0])

print("\nAfter")
print("Duplicates:", clean_df.duplicated().sum())
print("Missing Values:", clean_df.isnull().sum().sum())
print("Total Noise Values:", noise_count)
print("Dataset Size:", clean_df.shape)

Before
Duplicates: 50
Missing Values: 2075
Total Noise Values: 0
Dataset Size: (5050, 17)

After
Duplicates: 0
Missing Values: 0
Total Noise Values: 0
Dataset Size: (5000, 17)


In [44]:
import numpy as np
from sklearn.preprocessing import StandardScaler
#Copy Dataset
transform_df = clean_df.copy()
#Log transformation for highly skewed Purchase Amount
transform_df["Purchase Amount (USD)"] = np.log1p(
    transform_df["Purchase Amount (USD)"]
)
print("Before:",clean_df["Purchase Amount (USD)"].skew())
print("After:",np.log1p(clean_df["Purchase Amount (USD)"]).skew())
#Standardise numerical features
numeric_features = [
    "Age",
    "Purchase Amount (USD)",
    "Review Rating",
    "Previous Purchases"
]
scaler = StandardScaler()
transform_df[numeric_features] = scaler.fit_transform(transform_df[numeric_features])

display(transform_df[numeric_features].head())

Before: 3.574227835246709
After: 1.83078768728959


,Age,Purchase Amount (USD),Review Rating,Previous Purchases
0,-1.452986,-0.065096,-0.579632,0.788167
1,0.446085,0.173435,0.281159,-0.380332
2,-1.714927,-0.410840,-0.702602,-0.526394
3,-1.452986,0.045425,0.650069,-0.015176
4,-0.274252,0.118302,-0.087751,-0.599425


In [66]:
# Before feature selection & extraction
print("Before:", clean_df.shape[1], "features")
# Feature selection
feature_df = clean_df.drop(columns=["Customer ID"])
# After feature selection
print("After Feature Selection:", feature_df.shape[1], "features (Removed Customer ID)")
display(feature_df.head())
# Feature extraction
feature_df["Age Group"] = pd.cut(
    feature_df["Age"],
    bins=[17, 25, 40, 60, 100],
    labels=["18-25", "26-40", "41-60", "60+"]
)
# After feature extraction
print("After Feature Extraction:", feature_df.shape[1], "features (Added Age Group)")
display(
    feature_df[["Age", "Age Group"]].head()
)
# Encode target
feature_df["Subscription Status"] = feature_df[
    "Subscription Status"
].map({"No": 0, "Yes": 1})

# Encode categorical features
feature_df = pd.get_dummies(
    feature_df,
    dtype=int
)
# After categorical encoding 
print("After Encoding:", feature_df.shape[1], "features")

display(feature_df.head())

Before: 17 features
After Feature Selection: 16 features (Removed Customer ID)


,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Previous Purchases,Payment Method,Frequency of Purchases
0,22,Female,T-shirt,Clothing,68.0,California,XL,Olive,Winter,3.2,No,Standard,No,36.0,Cash,Weekly
1,51,Male,Sunglasses,Accessories,84.0,South Carolina,M,White,Spring,3.9,Yes,Free Shipping,Yes,20.0,Debit Card,Quarterly
2,18,Female,Shirt,Clothing,50.0,Montana,M,Black,Winter,3.1,No,2-Day Shipping,No,18.0,Cash,Monthly
3,22,Male,Gloves,Accessories,75.0,Illinois,L,Red,Fall,4.2,No,Store Pickup,No,25.0,Cash,Annually
4,40,Female,Jewelry,Accessories,80.0,Alabama,L,Yellow,Spring,3.6,No,Store Pickup,No,17.0,Credit Card,Weekly


After Feature Extraction: 17 features (Added Age Group)


,Age,Age Group
0,22,18-25
1,51,41-60
2,18,18-25
3,22,18-25
4,40,26-40


After Encoding: 156 features


,Age,Purchase Amount (USD),Review Rating,Subscription Status,Previous Purchases,Gender_Female,Gender_Male,Gender_Other,Item Purchased_Backpack,Item Purchased_Bag,...,Frequency of Purchases_Bi-Weekly,Frequency of Purchases_Every 3 Months,Frequency of Purchases_Fortnightly,Frequency of Purchases_Monthly,Frequency of Purchases_Quarterly,Frequency of Purchases_Weekly,Age Group_18-25,Age Group_26-40,Age Group_41-60,Age Group_60+
0,22,68.0,3.2,0,36.0,1,0,0,0,0,...,0,0,0,0,0,1,1,0,0,0
1,51,84.0,3.9,1,20.0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2,18,50.0,3.1,0,18.0,1,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0
3,22,75.0,4.2,0,25.0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,40,80.0,3.6,0,17.0,1,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0


In [71]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Features and target
X = feature_df.drop(columns=["Subscription Status"])
y = feature_df["Subscription Status"]

# Split data (Training & Testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Before SMOTE:")
print(y_train.value_counts())

# Implement SMOTE to training data
X_train, y_train = SMOTE(random_state=42).fit_resample(
    X_train, y_train
)

print("\nAfter SMOTE:")
print(y_train.value_counts())

Before SMOTE:
Subscription Status
0    2737
1    1263
Name: count, dtype: int64

After SMOTE:
Subscription Status
1    2737
0    2737
Name: count, dtype: int64
